# Sentence embeddings and similarity

[Back to the introduction](demo.ipynb). Run in order. MiniLM (384 dimensions) is the smaller English baseline; MPNet (768 dimensions) is a larger optional comparison. Each model is disposed before the next loads. Mean pooling with the attention mask and L2 normalization produce one flat sentence vector.

`embedding_backend` selects WASM/WebGPU independently of generation `backend`; `None` uses the runtime default. `embedding_dtype` pins weights without dtype fallback; `None` uses the runtime default. Offline-precomputed vectors must match the model snapshot, weights, tokenizer, input prefixes, pooling, and normalization.

In [ ]:
from lmlite import LLM

models_to_try = ["Xenova/all-MiniLM-L6-v2"]
# Opt in to the larger comparison before running the next cell:
# models_to_try.append("Xenova/all-mpnet-base-v2")
docs = [
    "JupyterLite runs entirely in the browser.",
    "Python is widely used for machine learning.",
    "TypeScript is useful for frontend applications.",
]
query = "Where does JupyterLite run?"

In [ ]:
for model_id in models_to_try:
    embedder = await LLM.create(
        embedding_model=model_id,
        embedding_backend="wasm",
        embedding_dtype="q8",
    )
    try:
        vector = await embedder.embed("Hello world")
        norm = sum(value * value for value in vector) ** 0.5
        print(f"{model_id}: dimension={len(vector)}, norm={norm:.6f}")
        for doc, score in await embedder.similarity_search(query, docs, k=2):
            print(f"{score:.3f} -> {doc}")
    finally:
        await embedder.dispose()

Downloaded weights remain cached after disposal. WebGPU trials should use a supported model/dtype combination; no device fallback is performed. Restart the kernel if memory remains allocated before further trials.